# Three small experiments with POS tagging

Companion notebook to [the post on mariaa.tech](https://mariaa.tech/blog/three-experiments-with-pos).

Three experiments in one notebook:

1. **Three taggers disagree** — nltk vs spaCy vs Stanza on the same English sentence.
2. **POS tags as features** — do POS-tag distributions help a text classifier vs plain TF-IDF?
3. **Error compounding** — what does the multiplicative cascade actually look like on real data?

**Estimated runtime:** ~60s on a fresh Colab runtime (Stanza install is the slow part).

## Setup

Run this cell once per session. Stanza needs an extra download for the English model — ~30s.

In [ ]:
!pip install -q nltk spacy stanza scikit-learn
!python -m spacy download -q en_core_web_sm

import nltk, spacy, stanza
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

nltk.download(["averaged_perceptron_tagger_eng", "punkt_tab"], quiet=True)

spacy_nlp = spacy.load("en_core_web_sm")

stanza.download("en", verbose=False)
stanza_nlp = stanza.Pipeline(lang="en", processors="tokenize,pos", verbose=False)

print("All three taggers ready.")

## Experiment 1 · Three taggers disagree

One moderately tricky English sentence, three taggers, look at the diff.

In [ ]:
sentence = "Visiting relatives can be boring."

nltk_tags  = nltk.pos_tag(nltk.word_tokenize(sentence))
spacy_tags = [(t.text, t.tag_) for t in spacy_nlp(sentence)]
stanza_tags = [(w.text, w.xpos) for s in stanza_nlp(sentence).sentences for w in s.words]

print(f"{'token':<12} {'nltk':<8} {'spaCy':<8} {'Stanza':<8}  diff?")
print("-" * 52)
for (a_w, a_t), (b_w, b_t), (c_w, c_t) in zip(nltk_tags, spacy_tags, stanza_tags):
    flag = "" if a_t == b_t == c_t else "  <-- diff"
    print(f"{a_w:<12} {a_t:<8} {b_t:<8} {c_t:<8}{flag}")

**Read it:** `boring` is the interesting one. nltk says VBG (gerund), spaCy and Stanza say JJ (adjective). Same eight letters, three different grammatical roles inferred from three different training corpora.

Try your own ambiguous sentences in the cell below.

In [ ]:
# Your turn: try sentences with structural ambiguity
# 'Time flies like an arrow.' — two valid readings
# 'I saw the man with the telescope.' — three for parsing
# 'Buffalo buffalo Buffalo buffalo buffalo buffalo Buffalo buffalo.' — the famous one

test = "Time flies like an arrow."
print("nltk  :", nltk.pos_tag(nltk.word_tokenize(test)))
print("spaCy :", [(t.text, t.tag_) for t in spacy_nlp(test)])
print("Stanza:", [(w.text, w.xpos) for s in stanza_nlp(test).sentences for w in s.words])

## Experiment 2 · Do POS tags help a classifier?

A small sentiment-style classification task. Compare three feature sets:

1. **TF-IDF only** — bag of words
2. **POS distribution only** — fraction of each tag type per document
3. **TF-IDF + POS** — concatenated

Tiny synthetic corpus so the notebook stays self-contained.

In [ ]:
# Tiny synthetic sentiment corpus — 24 short reviews
corpus = [
    # Positive reviews
    "This film is absolutely fantastic and the acting is brilliant.",
    "Loved every single minute of it. A masterpiece, truly inspiring.",
    "Hilarious and heartwarming. The cast was wonderful throughout.",
    "Best movie I have seen this year. Compelling, emotional, unforgettable.",
    "A stunning visual achievement and a powerful story. Highly recommend it.",
    "Wonderful direction, gorgeous cinematography, and a smart, witty script.",
    "This show was great. The characters were lovable and the plot tight.",
    "An excellent debut, charming and tightly paced. Highly enjoyable.",
    "Beautiful, thoughtful, and surprisingly funny. I will watch it again.",
    "Fantastic performance by the lead. Genuinely moving and well crafted.",
    "Loved the music, loved the writing. Easily one of the best of the year.",
    "A delight from start to finish. Smart, funny, and totally engaging.",
    # Negative reviews
    "Boring, predictable, and far too long. I wanted my time back.",
    "Terrible acting and a confusing plot. Not worth the ticket price.",
    "Slow, dull, and emotionally hollow. A disappointment on every level.",
    "Awful screenplay and wooden performances. I struggled to stay awake.",
    "The dialogue is clunky, the pacing is dreadful. Hard to sit through.",
    "Lazy writing and forgettable characters. A complete waste of an evening.",
    "This was painful. Bad direction, bad acting, bad everything.",
    "Cliched, badly edited, and not even slightly funny. Skip it entirely.",
    "Dull, dragging, and full of plot holes. I left halfway through.",
    "A boring, overlong mess. The script needed several more rewrites.",
    "Bland direction and a lifeless lead performance. Deeply forgettable.",
    "Bad film. Bad story. Bad acting. Bad music. Just bad.",
]
labels = [1] * 12 + [0] * 12   # 1 = positive, 0 = negative

X_train, X_test, y_train, y_test = train_test_split(
    corpus, labels, test_size=8, random_state=42, stratify=labels
)
print(f"Train: {len(X_train)} reviews · Test: {len(X_test)} reviews")

In [ ]:
# Feature set 1: TF-IDF
tfidf = TfidfVectorizer()
Xtr_tfidf = tfidf.fit_transform(X_train)
Xte_tfidf = tfidf.transform(X_test)

# Feature set 2: POS distribution (fraction of each tag per document)
def pos_dist(text, tag_vocab):
    tags = [t.tag_ for t in spacy_nlp(text)]
    total = max(len(tags), 1)
    return np.array([tags.count(t) / total for t in tag_vocab])

# Build the tag vocabulary from training data
train_tags_all = set()
for text in X_train:
    for t in spacy_nlp(text):
        train_tags_all.add(t.tag_)
tag_vocab = sorted(train_tags_all)
print(f"Tag vocabulary ({len(tag_vocab)}): {tag_vocab}")

Xtr_pos = np.array([pos_dist(t, tag_vocab) for t in X_train])
Xte_pos = np.array([pos_dist(t, tag_vocab) for t in X_test])

In [ ]:
from scipy.sparse import hstack, csr_matrix

# Feature set 3: TF-IDF + POS distribution (concatenated)
Xtr_both = hstack([Xtr_tfidf, csr_matrix(Xtr_pos)])
Xte_both = hstack([Xte_tfidf, csr_matrix(Xte_pos)])

def evaluate(name, Xtr, Xte):
    clf = LogisticRegression(max_iter=1000)
    clf.fit(Xtr, y_train)
    pred = clf.predict(Xte)
    acc = accuracy_score(y_test, pred)
    f1  = f1_score(y_test, pred)
    print(f"{name:<22} acc={acc:.3f}  f1={f1:.3f}")

print(f"{'features':<22} {'accuracy':<10} {'f1'}")
print("-" * 44)
evaluate("TF-IDF only",     Xtr_tfidf, Xte_tfidf)
evaluate("POS distribution", Xtr_pos, Xte_pos)
evaluate("TF-IDF + POS",     Xtr_both, Xte_both)

**Read it:** TF-IDF on its own is usually the strongest. POS distribution alone is much weaker (it has only ~30 features). Concatenating them sometimes lifts accuracy, sometimes doesn't — depends on the corpus and how much the bag-of-words signal already captures.

The post discusses the more nuanced **POS-as-joint-feature** approach (e.g. `good_JJ` and `good_RB` as separate tokens). Try implementing it in the cell below.

## Experiment 3 · Error compounding cascade

Simulate a four-stage pipeline (TOK → POS → DP → CLS) where each stage has its own error rate, and measure end-to-end accuracy.

Simple version: each stage is correct independently with probability `pᵢ`. End-to-end correctness = all stages correct = `∏ pᵢ`.

In [ ]:
import numpy as np
rng = np.random.default_rng(seed=42)

stages = {
    "Tokenization":     0.95,
    "POS tagging":      0.93,
    "Dependency parse": 0.90,
    "Classification":   0.85,
}

N = 10_000   # simulated sentences

# For each sentence, simulate whether each stage was correct
correct = np.ones(N, dtype=bool)
print(f"{'stage':<20} {'stage acc':<12} {'survivors after':<18}")
print("-" * 52)
for stage, p in stages.items():
    correct &= rng.random(N) < p
    print(f"{stage:<20} {p:<12} {correct.sum():,} / {N:,}  ({correct.mean():.3f})")

print()
print(f"Expected end-to-end (product of stage accs): {np.prod(list(stages.values())):.3f}")
print(f"Measured end-to-end (simulation)            : {correct.mean():.3f}")

**Read it:** the simulation lands within noise of the product. Each layer looks fine alone (85–95%), but the multiplication takes you down to roughly 68% end-to-end.

The honest version — that the errors are correlated (not independent) — would push the measured end-to-end below the product. Worth modelling if you have time.

In [ ]:
# Bonus: show the value of fixing the earliest stage
# vs improving the last stage by the same amount

baseline = {"TOK": 0.95, "POS": 0.93, "DP": 0.90, "CLS": 0.85}
p_base = np.prod(list(baseline.values()))

# +1pp on earliest layer
earliest = baseline.copy(); earliest["TOK"] += 0.01
p_earliest = np.prod(list(earliest.values()))

# +1pp on last layer
latest = baseline.copy(); latest["CLS"] += 0.01
p_latest = np.prod(list(latest.values()))

print(f"Baseline                          : {p_base:.4f}")
print(f"+1pp on tokenisation (earliest)   : {p_earliest:.4f}  (gain {p_earliest - p_base:.4f})")
print(f"+1pp on classifier (latest)       : {p_latest:.4f}   (gain {p_latest - p_base:.4f})")

**Read it:** +1 percentage point on tokenisation buys you slightly more than +1pp on the classifier. The compounding amplifies improvements at the earliest stage. **Optimise from the bottom up.**

## Wrap-up

Three short experiments. Three small but real things to take away:

1. **Taggers disagree.** Pick the one whose training corpus matches your data.
2. **POS features help, marginally.** Bag of words already captures most of the signal in English.
3. **Errors compound.** A 95% layer × four times is 81%. The earliest layer matters most.

See [the full post](https://mariaa.tech/blog/three-experiments-with-pos) for the longer write-up.